# War Supply Problem in Pure Python
## Exact Techniques for Book Problem 3.1

This notebook solves the war-supply model from book section `3.1` with exact Python techniques.

We combine:
- reduced exhaustive enumeration over the armament-equality patterns,
- memoized recursive search over the same feasible flight plans,
- explicit verification of the published textbook plan.

The book reports a minimum mission cost of `4,820,000` USD. Because both aircraft types have the same cost per transported ton on each destination, the model has multiple optimal flight plans. The final section keeps the published plan as one optimal reference.


In [1]:
from __future__ import annotations

from functools import lru_cache, wraps
from time import perf_counter


In [2]:
def timed(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        result = func(*args, **kwargs)
        elapsed = perf_counter() - start
        return result, elapsed

    return wrapper


COST = {
    ("tip1", "A"): 120_000,
    ("tip1", "B"): 80_000,
    ("tip2", "A"): 90_000,
    ("tip2", "B"): 60_000,
}
CAPACITY = {
    "tip1": {"armament": 20, "vehicles": 4, "jeeps": 3},
    "tip2": {"armament": 15, "vehicles": 1, "jeeps": 2},
}
DEMAND_A = {"armament": 570, "vehicles": 25, "jeeps": 18}
DEMAND_B_ARMAMENT = 350
AIRPORT_STOCK = {"armament": 2_000, "vehicles": 150, "jeeps": 230}
EXPECTED_OBJECTIVE = 4_820_000


def build_plan(tip1_to_A, tip2_to_A, tip1_to_B, tip2_to_B):
    armament_A = 20 * tip1_to_A + 15 * tip2_to_A
    vehicles_A = 4 * tip1_to_A + tip2_to_A
    jeeps_A = 3 * tip1_to_A + 2 * tip2_to_A
    armament_B = 20 * tip1_to_B + 15 * tip2_to_B
    vehicles_B = 4 * tip1_to_B + tip2_to_B
    jeeps_B = 3 * tip1_to_B + 2 * tip2_to_B

    if armament_A < DEMAND_A["armament"] or vehicles_A < DEMAND_A["vehicles"] or jeeps_A < DEMAND_A["jeeps"]:
        return None
    if armament_B < DEMAND_B_ARMAMENT:
        return None
    if vehicles_B / 120 + jeeps_B / 90 > 1 + 1e-9:
        return None
    if (tip1_to_A + tip1_to_B) / 120 + (tip2_to_A + tip2_to_B) / 85 > 1 + 1e-9:
        return None

    used_stock = {
        "armament": armament_A + armament_B,
        "vehicles": vehicles_A + vehicles_B,
        "jeeps": jeeps_A + jeeps_B,
    }
    if any(used_stock[item] > AIRPORT_STOCK[item] for item in AIRPORT_STOCK):
        return None

    return {
        "tip1_to_A": tip1_to_A,
        "tip2_to_A": tip2_to_A,
        "tip1_to_B": tip1_to_B,
        "tip2_to_B": tip2_to_B,
        "cost": (
            COST[("tip1", "A")] * tip1_to_A
            + COST[("tip2", "A")] * tip2_to_A
            + COST[("tip1", "B")] * tip1_to_B
            + COST[("tip2", "B")] * tip2_to_B
        ),
        "deliveries": {
            "A": {"armament": armament_A, "vehicles": vehicles_A, "jeeps": jeeps_A},
            "B": {"armament": armament_B, "vehicles": vehicles_B, "jeeps": jeeps_B},
        },
        "airport_usage": used_stock,
    }


def flight_combinations_for_exact_armament(target_armament):
    combinations = []
    for tip2 in range(0, target_armament // 15 + 1):
        remaining = target_armament - 15 * tip2
        if remaining >= 0 and remaining % 20 == 0:
            combinations.append((remaining // 20, tip2))
    return combinations


def choose_better(left, right):
    if left is None:
        return right
    if right is None:
        return left
    left_key = (left["cost"], left["tip1_to_A"], left["tip2_to_A"], left["tip1_to_B"], left["tip2_to_B"])
    right_key = (right["cost"], right["tip1_to_A"], right["tip2_to_A"], right["tip1_to_B"], right["tip2_to_B"])
    return right if right_key < left_key else left


## 1. Reduced Exhaustive Search

Since every aircraft must travel fully loaded, and both aircraft types have the same transportation cost per armament ton on each destination, every optimal plan can be searched on the exact-armament equalities:

- point `A`: `20 * tip1 + 15 * tip2 = 570`,
- point `B`: `20 * tip1 + 15 * tip2 = 350`.

That collapses the full integer search to a handful of candidate flight combinations.


In [3]:
@timed
def solve_war_supply_reduced_search():
    best = None
    for tip1_to_A, tip2_to_A in flight_combinations_for_exact_armament(570):
        for tip1_to_B, tip2_to_B in flight_combinations_for_exact_armament(350):
            best = choose_better(best, build_plan(tip1_to_A, tip2_to_A, tip1_to_B, tip2_to_B))
    return best


reduced_result, reduced_time = solve_war_supply_reduced_search()
print("Reduced-search result:", reduced_result)
print(f"Elapsed time: {reduced_time:.6f} seconds")
assert reduced_result["cost"] == EXPECTED_OBJECTIVE


Reduced-search result: {'tip1_to_A': 0, 'tip2_to_A': 38, 'tip1_to_B': 1, 'tip2_to_B': 22, 'cost': 4820000, 'deliveries': {'A': {'armament': 570, 'vehicles': 38, 'jeeps': 76}, 'B': {'armament': 350, 'vehicles': 26, 'jeeps': 47}}, 'airport_usage': {'armament': 920, 'vehicles': 64, 'jeeps': 123}}
Elapsed time: 0.000084 seconds


## 2. Memoized Recursive Search

The same exact-armament candidate sets can also be explored recursively. The recursive version is useful as a clean, reusable feasibility checker when the flight pools grow.


In [4]:
ARMAMENT_CHOICES_A = tuple(flight_combinations_for_exact_armament(570))
ARMAMENT_CHOICES_B = tuple(flight_combinations_for_exact_armament(350))


@timed
def solve_war_supply_memoized():
    @lru_cache(maxsize=None)
    def best_b(index, tip1_to_A, tip2_to_A):
        if index == len(ARMAMENT_CHOICES_B):
            return None
        tip1_to_B, tip2_to_B = ARMAMENT_CHOICES_B[index]
        current = build_plan(tip1_to_A, tip2_to_A, tip1_to_B, tip2_to_B)
        tail = best_b(index + 1, tip1_to_A, tip2_to_A)
        return choose_better(current, tail)

    best = None
    for tip1_to_A, tip2_to_A in ARMAMENT_CHOICES_A:
        best = choose_better(best, best_b(0, tip1_to_A, tip2_to_A))
    return best


memoized_result, memoized_time = solve_war_supply_memoized()
print("Memoized result:", memoized_result)
print(f"Elapsed time: {memoized_time:.6f} seconds")
assert memoized_result["cost"] == EXPECTED_OBJECTIVE


Memoized result: {'tip1_to_A': 0, 'tip2_to_A': 38, 'tip1_to_B': 1, 'tip2_to_B': 22, 'cost': 4820000, 'deliveries': {'A': {'armament': 570, 'vehicles': 38, 'jeeps': 76}, 'B': {'armament': 350, 'vehicles': 26, 'jeeps': 47}}, 'airport_usage': {'armament': 920, 'vehicles': 64, 'jeeps': 123}}
Elapsed time: 0.000083 seconds


## 3. Published Textbook Plan

The book reports one optimal flight plan even though the model admits multiple optima. We keep that reference plan here and verify that it is feasible and reaches the same minimum cost.


In [5]:
textbook_result = build_plan(21, 10, 10, 10)
print("Textbook plan:", textbook_result)

assert textbook_result is not None
assert textbook_result["cost"] == EXPECTED_OBJECTIVE
assert reduced_result["cost"] == memoized_result["cost"] == textbook_result["cost"]

print("\nAll Python techniques agree on the minimum mission cost.")
print("Reduced-search optimal plan ->", reduced_result)
print("Published reference plan   ->", textbook_result)


Textbook plan: {'tip1_to_A': 21, 'tip2_to_A': 10, 'tip1_to_B': 10, 'tip2_to_B': 10, 'cost': 4820000, 'deliveries': {'A': {'armament': 570, 'vehicles': 94, 'jeeps': 83}, 'B': {'armament': 350, 'vehicles': 50, 'jeeps': 50}}, 'airport_usage': {'armament': 920, 'vehicles': 144, 'jeeps': 133}}

All Python techniques agree on the minimum mission cost.
Reduced-search optimal plan -> {'tip1_to_A': 0, 'tip2_to_A': 38, 'tip1_to_B': 1, 'tip2_to_B': 22, 'cost': 4820000, 'deliveries': {'A': {'armament': 570, 'vehicles': 38, 'jeeps': 76}, 'B': {'armament': 350, 'vehicles': 26, 'jeeps': 47}}, 'airport_usage': {'armament': 920, 'vehicles': 64, 'jeeps': 123}}
Published reference plan   -> {'tip1_to_A': 21, 'tip2_to_A': 10, 'tip1_to_B': 10, 'tip2_to_B': 10, 'cost': 4820000, 'deliveries': {'A': {'armament': 570, 'vehicles': 94, 'jeeps': 83}, 'B': {'armament': 350, 'vehicles': 50, 'jeeps': 50}}, 'airport_usage': {'armament': 920, 'vehicles': 144, 'jeeps': 133}}


## Note on Multiple Optima

The exact cost depends only on the transported armament tons once the minimum requirements are met, so several flight combinations are optimal. The book's plan is therefore one optimal reference, not the only one.
